# Point 3 — Energy Barriers and Quasi-Ergodicity

The Contrastive Divergence (CD) algorithm does not easily switch between
different digit representations during Gibbs sampling — the chain tends to
stay trapped near the starting digit class. This notebook investigates
the **energy barriers** that explain this quasi-ergodic behavior.

We:

1. Train a Hinton-faithful RBM (n_h = 500, SGD with momentum, CD-1);
2. Design a **forced Gibbs transition** (modified CD) that gradually pushes
   the visible state from one digit to another, measuring the joint energy
   $E(v, h)$ along the way;
3. Compare with unmodified (natural) CD to quantify quasi-ergodicity.

> Run cells **top to bottom**. The notebook is self-contained.

## 0. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14
})

from scipy.special import logit
from sklearn.datasets import fetch_openml

from rbm import RBM

## 1. Data Loading and Preprocessing

Same setup as the other notebooks: digits 0/1/2, pixels in `[0, 1]`,
small Gaussian noise for regularization.

In [ ]:
X_original, Y_original = fetch_openml("mnist_784", version=1,
                                      return_X_y=True, as_frame=False)

X_original = np.array(X_original, dtype=np.float32)
Y_original = np.array(Y_original, dtype=int)

mask = (Y_original == 0) | (Y_original == 1) | (Y_original == 2)
X_filtered = X_original[mask]
Y_filtered = Y_original[mask]

def preprocess_mnist(images):
    images = images / 255.0
    images = images + np.random.normal(0, 0.01, images.shape)
    return np.clip(images, 0, 1)

X_train = preprocess_mnist(X_filtered)

print(f"Filtered 0,1,2 : {X_filtered.shape}")
print(f"Range : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Mean  : {X_train.mean():.3f}")

## 2. Model Training

We train a moderately large RBM (`n_h = 500`) following Hinton's
recommended recipe: SGD with momentum (0.5 → 0.9), a learning-rate
schedule starting at 0.1 and decaying to 0.01, L2 weight decay
$\lambda = 2 \times 10^{-4}$, and CD-1. The visible biases are
initialized manually via `logit(p_i)` (Hinton, Section 8).

We use an explicit mini-batch loop instead of `RBM.train()` to have
finer control and per-epoch diagnostics (update ratio, max weight, etc.).

In [ ]:
# --- RBM creation ---
rbm = RBM(
    n_v=784,
    n_h=500,
    lr=0.01,
    lr_schedule="hinton",     # 0.1 for first epochs, then lr
    momentum=0.9,
    weight_decay=0.0002,
    optimizer="sgd",
    spins=False,
    gamma=0.0,
)

# --- Manual visible-bias initialization (Hinton Sec. 8) ---
p = np.clip(X_train.mean(axis=0), 0.01, 0.99)
rbm.v_bias = np.log(p / (1 - p))
print(f"Visible biases initialized  range: [{rbm.v_bias.min():.3f}, {rbm.v_bias.max():.3f}]")

In [ ]:
# --- Training loop ---
n_epochs   = 20
batch_size = 100
n_samples  = X_train.shape[0]

losses, sparsity_list, update_ratios, weights_max_list = [], [], [], []

print(f"Training — {n_epochs} epochs, batch size {batch_size}, "
      f"{n_samples // batch_size} batches/epoch\n")

for epoch in range(1, n_epochs + 1):
    epoch_metrics = []

    current_lr  = rbm._get_lr(epoch, Nepoch=n_epochs, lr_fin=rbm.lr)
    current_mom = rbm._get_momentum(epoch)

    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    X_shuffled = X_train[indices]

    for start in range(0, n_samples, batch_size):
        batch = X_shuffled[start : start + batch_size]
        metrics = rbm.update(batch, n_cd=1, persistent=False,
                             lr=current_lr, momentum=current_mom)
        epoch_metrics.append(metrics)

    losses.append(np.mean([m["mse"]          for m in epoch_metrics]))
    sparsity_list.append(np.mean([m["sparsity"]     for m in epoch_metrics]))
    update_ratios.append(np.mean([m["update_ratio"]  for m in epoch_metrics]))
    weights_max_list.append(np.mean([m["weights_max"] for m in epoch_metrics]))

    print(f"Epoch {epoch:2d}/{n_epochs}  MSE={losses[-1]:.5f}  "
          f"ratio={update_ratios[-1]:.5f}  sparsity={sparsity_list[-1]:.3f}  "
          f"W_max={weights_max_list[-1]:.3f}")

print("\nTraining complete.")

## 3. Forced Gibbs Transition (Modified CD)

### Methodology

To measure the energy barrier separating different digit representations,
we design a **forced Gibbs transition** algorithm:

1. Start from a typical configuration $v_0$ of digit "1".
2. Define a target configuration $v_{\text{target}}$ (a typical digit "2").
3. Run Gibbs sampling with **gradual forcing** over $N$ steps:
   - sample $h \sim P(h \mid v_t)$ and $v' \sim P(v \mid h)$ as in standard CD;
   - apply forcing: $v_{t+1} = (1 - \alpha_t)\, v' + \alpha_t\, v_{\text{target}}$;
   - $\alpha_t$ increases linearly from 0 to 1.
4. Measure the **joint energy** $E(v_t, h_t)$ at each intermediate step.

The energy profile reveals a peak (the barrier) at configurations that
are "between" the two digit classes.

In [ ]:
# --- Select representative start / target digits ---
idx_1 = np.where(Y_filtered == 1)[0][0]
idx_2 = np.where(Y_filtered == 2)[0][0]
v_start  = X_train[idx_1].copy()
v_target = X_train[idx_2].copy()

# --- Forced transition 1 → 2 ---
n_steps = 50
forcing_schedule = np.linspace(0, 1, n_steps)

energies, v_trajectory, h_trajectory = [], [], []
v_current = v_start[np.newaxis, :].copy()

print("FORCED GIBBS TRANSITION: Digit 1 -> Digit 2")
print("=" * 70)

for step, alpha in enumerate(forcing_schedule):
    # Standard Gibbs step
    _, h_state = rbm.sample_h(v_current)
    v_prob, v_state = rbm.sample_v(h_state)

    # Forcing toward target
    v_forced = (1 - alpha) * v_state + alpha * v_target[np.newaxis, :]

    # Measure joint energy E(v, h)
    energy = rbm.energy(v_forced[0], h_state[0])

    energies.append(energy)
    v_trajectory.append(v_forced[0].copy())
    h_trajectory.append(h_state[0].copy())

    v_current = v_forced.copy()

    if step % 10 == 0:
        print(f"  step {step:2d}  alpha={alpha:.2f}  E={energy:8.2f}")

# --- Barrier analysis ---
E_initial = energies[0]
E_final   = energies[-1]
E_max     = max(energies)
E_max_idx = energies.index(E_max)
barrier_height = E_max - E_initial

print("=" * 70)
print(f"\nENERGY BARRIER ANALYSIS:")
print(f"  E(digit 1, start)     = {E_initial:8.2f}")
print(f"  E(peak, step {E_max_idx:2d})      = {E_max:8.2f}")
print(f"  E(digit 2, end)       = {E_final:8.2f}")
print(f"  Barrier height (DE)   = {barrier_height:8.2f}")
print(f"  Energy difference 1->2= {E_final - E_initial:8.2f}")

In [ ]:
# --- Visualization: energy profile + snapshots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Energy profile
ax = axes[0, 0]
ax.plot(forcing_schedule, energies, "o-", lw=2, ms=4, color="darkblue")
ax.axhline(E_initial, color="green", ls="--", alpha=0.7, label="E(digit 1)")
ax.axhline(E_final,   color="red",   ls="--", alpha=0.7, label="E(digit 2)")
ax.axhline(E_max,     color="orange", ls=":",  alpha=0.7,
           label=f"Peak (DE={barrier_height:.1f})")
ax.fill_between(forcing_schedule, E_initial, energies, alpha=0.2, color="blue")
ax.set(xlabel="Forcing strength alpha", ylabel="Joint Energy E(v,h)",
       title="Energy Barrier during Forced Transition")
ax.legend(); ax.grid(True, alpha=0.3)

# Start digit
ax = axes[0, 1]
ax.imshow(v_start.reshape(28, 28), cmap="gray")
ax.set_title(f"Start: Digit 1\nE = {E_initial:.2f}"); ax.axis("off")

# Peak configuration
ax = axes[1, 0]
ax.imshow(v_trajectory[E_max_idx].reshape(28, 28), cmap="gray")
ax.set_title(f"Peak (step {E_max_idx})\nE = {E_max:.2f}"); ax.axis("off")

# End digit
ax = axes[1, 1]
ax.imshow(v_trajectory[-1].reshape(28, 28), cmap="gray")
ax.set_title(f"End: Digit 2\nE = {E_final:.2f}"); ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Natural CD Behavior: Why Standard CD Cannot Cross the Barrier

We compare the forced-transition energy barrier with the typical energy
fluctuations of an **unmodified** Gibbs chain starting from digit "1".
If the barrier is much larger than the thermal fluctuations, standard CD
cannot spontaneously cross it — hence quasi-ergodicity.

In [ ]:
# --- Standard CD chain from digit 1 for 100 steps ---
v_cd = v_start[np.newaxis, :].copy()
cd_energies = []

for _ in range(100):
    _, h = rbm.sample_h(v_cd)
    _, v_cd = rbm.sample_v(h)
    cd_energies.append(rbm.energy(v_cd[0], h[0]))

print("Standard CD — 100 Gibbs steps from digit 1:")
print(f"  Energy range       : [{min(cd_energies):.2f}, {max(cd_energies):.2f}]")
print(f"  Mean energy        : {np.mean(cd_energies):.2f}")
print(f"  Std (fluctuation)  : {np.std(cd_energies):.2f}")
print(f"\n  Barrier height     : {barrier_height:.2f}")
print(f"  Ratio barrier/std  : {barrier_height / np.std(cd_energies):.1f}x")
print("\n-> The barrier is much higher than typical thermal fluctuations,")
print("   explaining why CD gets stuck (quasi-ergodic behavior).")

## 5. Natural Transition Probabilities

We launch 50 independent Gibbs chains (1000 steps each) from digit 1
and from digit 2, and classify each visited state using nearest-prototype
classification. Low transition probabilities confirm CD stays trapped.

In [ ]:
# --- Helper functions ---
def compute_prototypes(data, labels):
    """Mean image per class -> prototypes for nearest-neighbor classification."""
    return {int(d): data[labels == d].mean(axis=0)
            for d in np.unique(labels)}

def classify_nearest(v, prototypes):
    """Assign v to the closest prototype (L2 distance)."""
    dists = {d: np.sum((v - p)**2) for d, p in prototypes.items()}
    return min(dists, key=dists.get)

def measure_natural_transitions(rbm, v_start, prototypes,
                                n_chains=50, n_steps=1000):
    """Fraction of chains that visit each digit class during natural CD."""
    start_digit = classify_nearest(v_start, prototypes)
    transitions = {d: 0 for d in prototypes}

    for _ in range(n_chains):
        v = v_start[np.newaxis, :].copy()
        visited = {start_digit}
        for _ in range(n_steps):
            _, h = rbm.sample_h(v)
            _, v = rbm.sample_v(h)
            visited.add(classify_nearest(v[0], prototypes))
        for d in visited:
            if d != start_digit:
                transitions[d] += 1

    return {d: count / n_chains for d, count in transitions.items()}

# --- Compute ---
prototypes = compute_prototypes(X_train, Y_filtered)

v_1 = X_train[np.where(Y_filtered == 1)[0][0]]
v_2 = X_train[np.where(Y_filtered == 2)[0][0]]

print("Natural CD transitions (50 chains x 1000 steps):")
print("=" * 55)

print("\nStarting from digit 1:")
trans_1 = measure_natural_transitions(rbm, v_1, prototypes)
for d, prob in sorted(trans_1.items()):
    if d != 1:
        print(f"  -> digit {d}: {prob:5.1%}")

print("\nStarting from digit 2:")
trans_2 = measure_natural_transitions(rbm, v_2, prototypes)
for d, prob in sorted(trans_2.items()):
    if d != 2:
        print(f"  -> digit {d}: {prob:5.1%}")

print("=" * 55)
print("\nLow probabilities confirm CD is trapped by the energy barriers.")

## 6. Summary Figure

Three-panel summary: (A) the energy profile of the forced transition;
(B) energy fluctuations of natural CD (trapped); (C) bar chart of
transition probabilities from digit 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (A) Forced transition energy profile
ax = axes[0]
ax.plot(forcing_schedule, energies, "o-", lw=2)
ax.axhline(E_max, color="red", ls=":",
           label=f"Barrier: DE={barrier_height:.1f}")
ax.set(title="(A) Forced Transition\nEnergy Profile",
       xlabel="Forcing alpha", ylabel="Energy E(v,h)")
ax.legend(); ax.grid(True, alpha=0.3)

# (B) Natural CD energy trace
ax = axes[1]
ax.plot(cd_energies, lw=1, alpha=0.7, color="blue")
ax.axhline(np.mean(cd_energies), color="darkblue", ls="--",
           label=f"Mean +/- std: {np.mean(cd_energies):.1f} +/- {np.std(cd_energies):.1f}")
ax.set(title="(B) Natural CD\n(remains trapped)",
       xlabel="Gibbs steps", ylabel="Energy E(v,h)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (C) Transition probabilities from digit 1
ax = axes[2]
digits = sorted(trans_1.keys())
probs  = [trans_1[d] for d in digits]
colors = ["green" if d == 1 else "red" if d == 2 else "gray" for d in digits]
ax.bar([str(d) for d in digits], probs, color=colors, alpha=0.7)
ax.set(title="(C) Natural Transitions\n(1000 steps from digit 1)",
       xlabel="Target digit", ylabel="Probability", ylim=[0, 1])
ax.axhline(0.5, color="black", ls=":", alpha=0.3)

plt.tight_layout()
plt.show()